# Animate the hybridisation solutions in the complex plane
For a 2×2 pairs system (g/2π = -40 kHz), sweeps the frequency and follows, as arrows in the complex plane, the deranged determinant det(**D**), the adjugate entries adj(**D**)ₓ₁ₓ₁ and adj(**D**)ₓ₁ᵧ₁, the whole adjugate matrix, and the hybridisation solutions h_jk = adj(**D**)_jk / det(**D**).

Runtime: about 1-2 min by default, which shows the last frame of each sweep as a static figure. Set `MAKE_ANIMATIONS = True` to write the five animations to `output/` (about 1.5 h; the last one has 3001 frames).

# START

# Determinant
$\det(\mathbf{D})$

In [ ]:
from pathlib import Path
import matplotlib
import matplotlib.animation
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

# --- run settings ---------------------------------------------------------------------------------
# False (default): every animation is stepped through its frames without being rendered and shown as
# one static figure of its last frame (with every trail), which takes seconds.
# True: every animation is also written to output/ (GIF, or MP4 when ffmpeg is installed), which
# takes minutes. output/ is not tracked by git.
MAKE_ANIMATIONS = False
# keep every n-th frame; n > 1 gives a quick test of the full run
FRAME_STRIDE = 1
# resolution of the written animation files
ANIMATION_DPI = 75
OUTPUT_DIR = Path('output')


class SweepAnimation:
    """
    A frequency-sweep animation, update(frame) over num_frames frames, rendered only on request

    show(name) writes it to OUTPUT_DIR when MAKE_ANIMATIONS is True, otherwise it just runs
    update over the frames; either way it then displays the figure, which holds the last frame.
    """
    def __init__(self, fig, update, num_frames, fps=20):
        self.fig, self.update, self.fps = fig, update, fps
        self.frames = list(range(0, num_frames, FRAME_STRIDE))
        # closed so that the notebook does not also show the empty figure when the cell ends
        plt.close(fig)

    def show(self, name):
        if MAKE_ANIMATIONS:
            OUTPUT_DIR.mkdir(exist_ok=True)
            writer = 'ffmpeg' if matplotlib.animation.writers.is_available('ffmpeg') else 'pillow'
            path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
            animation = matplotlib.animation.FuncAnimation(self.fig, self.update, frames=self.frames, blit=False)
            animation.save(path, writer=writer, fps=self.fps, dpi=ANIMATION_DPI)
            print(f'wrote {path} ({len(self.frames)} frames)')
        else:
            for frame in self.frames:
                self.update(frame)
        self.fig.canvas.draw()
        display(self.fig)


gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
toy_params = dict(Ω_model='pairs', toy_gx=-40*kHz, toy_gy=-40*kHz)

system = '2×2'
Nd = system.split('×')
N, d = int(Nd[0]), int(Nd[1])
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=np.arange(0, 400 + 1, 1) * kHz,
    gas_kwargs=gas_kwargs,
    cavity_kwargs=cavity_kwargs,
    toy_params=toy_params,
    generate_power_series=True,
)

Det_fig = plt.figure(layout='constrained', figsize=(4, 4), dpi=75)
Det_ax_dict = Det_fig.subplot_mosaic([
    ['Det'],
])

marker = 'o'
colours = colours_50
lim = 10
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(Det_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color=black, linewidth=0.5)
    ax.axvline(0, color=black, linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])


ω_input = np.linspace(200, 350, 300, endpoint=True) * kHz
def update(frame):
    ω = ω_input[frame]
    for ii, ax in enumerate(list(Det_ax_dict.values())):
        fω = mtrap.deranged_determinant(ω)
        arrows[ii].set_label(f'Det(ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (Det_fig, *arrows, *output_trails)

ani = SweepAnimation(Det_fig, update, len(ω_input))

ani.show('determinant')

# Diagonal adjugate matrix entry
$\text{adj}(\mathbf{D})_{x_1 x_1}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
toy_params = dict(Ω_model='pairs', toy_gx=-40*kHz, toy_gy=-40*kHz)

system = '2×2'
Nd = system.split('×')
N, d = int(Nd[0]), int(Nd[1])
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=np.arange(0, 400 + 1, 1) * kHz,
    gas_kwargs=gas_kwargs,
    cavity_kwargs=cavity_kwargs,
    toy_params=toy_params,
    generate_power_series=True,
)

fig = plt.figure(layout='constrained', figsize=(4, 4), dpi=75)
ax_dict = fig.subplot_mosaic([
    ['Det'],
])

marker = 'o'
colours = colours_50
lim = 10
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color=black, linewidth=0.5)
    ax.axvline(0, color=black, linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])


ω_input = np.linspace(200, 350, 300, endpoint=True) * kHz

ij = 'x1x1'
# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(ax_dict.values())):
        fω = mtrap.deranged_adjugate_matrix_entry(ij, ω)
        arrows[ii].set_label(f'Adj[{ij}](ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (fig, *arrows, *output_trails)

ani = SweepAnimation(fig, update, len(ω_input))
ani.show('adjugate_x1x1')


# Off-diagonal adjugate matrix entry
$\text{adj}(\mathbf{D})_{x_1 y_1}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
toy_params = dict(Ω_model='pairs', toy_gx=-40*kHz, toy_gy=-40*kHz)

system = '2×2'
Nd = system.split('×')
N, d = int(Nd[0]), int(Nd[1])
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=np.arange(0, 400 + 1, 1) * kHz,
    gas_kwargs=gas_kwargs,
    cavity_kwargs=cavity_kwargs,
    toy_params=toy_params,
    generate_power_series=True,
)

fig = plt.figure(layout='constrained', figsize=(4, 4), dpi=75)
ax_dict = fig.subplot_mosaic([
    ['Det'],
])

marker = 'o'
colours = colours_50
lim = 10
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color=black, linewidth=0.5)
    ax.axvline(0, color=black, linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])


ω_input = np.linspace(200, 350, 300, endpoint=True) * kHz

ij = 'x1y1'
# Animation function
def update(frame):
    ω = ω_input[frame]

    # for ii, term in enumerate(terms):
    for ii, ax in enumerate(list(ax_dict.values())):
        fω = mtrap.deranged_adjugate_matrix_entry(ij, ω)
        arrows[ii].set_label(f'Adj[{ij}](ω)={fω:.3f}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'ω={ω / kHz:.1f} kHz')

    return (fig, *arrows, *output_trails)

ani = SweepAnimation(fig, update, len(ω_input))
ani.show('adjugate_x1y1')


# Grid: adjugate matrix
$\mathrm{adj}(\mathbf{D})$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
toy_params = dict(Ω_model='pairs', toy_gx=-40*kHz, toy_gy=-40*kHz)

system = '2×2'
Nd = system.split('×')
N, d = int(Nd[0]), int(Nd[1])
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=np.arange(0, 400 + 1, 1) * kHz,
    gas_kwargs=gas_kwargs,
    cavity_kwargs=cavity_kwargs,
    toy_params=toy_params,
    generate_power_series=True,
)

all_ij = []
for dim_j in list(mtrap.dimensions.values()):
    for dim_k in list(mtrap.dimensions.values()):

        jn, km = dim_j.label, dim_k.label
        all_ij.append(f'{jn}{km}')


fig = plt.figure(layout='constrained', figsize=(7, 7), dpi=75)
ax_dict = fig.subplot_mosaic([
    [ij for ij in all_ij[0:4]],
    [ij for ij in all_ij[4:8]],
    [ij for ij in all_ij[8:12]],
    [ij for ij in all_ij[12:16]],
])

marker = 'o'
colours = colours_50
lim = 5
arrows = []
output_trails = []
trails = []
for ii, ax in enumerate(list(ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color=black, linewidth=0.5)
    ax.axvline(0, color=black, linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])
ω_input = np.linspace(250, 350, 200, endpoint=True) * kHz

# Animation function
def update(frame):
    ω = ω_input[frame]
    for ii, (ij, ax) in enumerate(list(ax_dict.items())):
        fω = mtrap.deranged_adjugate_matrix_entry(ij, ω)
        arrows[ii].set_label(f'{ij}')
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.legend(loc='upper right')
        ax.set_title(f'{fω:.3f}')

    fig.suptitle(f'adj(ω); ω={ω / kHz:.1f} kHz')
    return (fig, *arrows, *output_trails)

ani = SweepAnimation(fig, update, len(ω_input))
ani.show('adjugate_grid')


# Grid: hybridisation solutions
$h_{jk} \equiv \frac{\mathrm{adj}(\mathbf{D})_{ij}}{\det(\mathbf{D})}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
toy_params = dict(Ω_model='pairs', toy_gx=-40*kHz, toy_gy=-40*kHz)

system = '2×2'
Nd = system.split('×')
N, d = int(Nd[0]), int(Nd[1])
mtrap = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=d,
    ω_range=np.arange(0, 400 + 1, 1) * kHz,
    gas_kwargs=gas_kwargs,
    cavity_kwargs=cavity_kwargs,
    toy_params=toy_params,
    generate_power_series=True,
)

all_ij = []
for dim_j in list(mtrap.dimensions.values()):
    for dim_k in list(mtrap.dimensions.values()):

        jn, km = dim_j.label, dim_k.label
        all_ij.append(f'{jn}{km}')

hjk_fig = plt.figure(layout='constrained', figsize=(8, 8), dpi=75)
hjk_ax_dict = hjk_fig.subplot_mosaic([
    [ij for ij in all_ij[0:4]],
    [ij for ij in all_ij[4:8]],
    [ij for ij in all_ij[8:12]],
    [ij for ij in all_ij[12:16]],
])

marker = 'o'
colours = colours_50
lim = 50
arrows, output_trails, trails = [], [], []
for ii, ax in enumerate(list(hjk_ax_dict.values())):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.axhline(0, color=black, linewidth=0.5)
    ax.axvline(0, color=black, linewidth=0.5)
    ax.grid(True, linestyle='--', alpha=0.7)

    arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
    output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
    trails.append([])

arrows = sum(arrows, [])

ω_input = np.arange(280, 310 + 0.01, 0.01) * kHz
det = np.array([mtrap.deranged_determinant(ω) for ω in ω_input])

def update(frame):
    ω = ω_input[frame]
    det_ω = det[frame]
    for ii, (ij, ax) in enumerate(list(hjk_ax_dict.items())):
        fω = mtrap.deranged_adjugate_matrix_entry(ij, ω) / det_ω
        arrows[ii].set_data([0, fω.real], [0, fω.imag])
        trails[ii].append((fω.real, fω.imag))
        output_trails[ii].set_offsets(trails[ii])
        ax.set_title(f'$h_{{{ij}}}(\omega) = ${fω:.3f}')

    hjk_fig.suptitle(f'ω={ω / kHz:.1f} kHz')
    return (hjk_fig, *arrows, *output_trails)

ani = SweepAnimation(hjk_fig, update, len(ω_input))
ani.show('hybridisation_solutions_grid')


# END